# 13 Colab Joint Generation + Contrastive Training

Run-all Colab notebook for optional Stage 4 joint generation + contrastive training after the generation pipeline has produced checkpoints.


## 1. Runtime And Drive Mount

This notebook is designed for Colab Run all. It clones the repository into `/content/neurovlm`, copies your generated data from Google Drive into the clone, and writes run outputs back to Drive.


In [ ]:
import os, sys, json, time, platform, subprocess, shutil
from pathlib import Path

print('Python:', sys.version)
print('Platform:', platform.platform())
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(gpu.stdout if gpu.returncode == 0 else 'No GPU detected by nvidia-smi. Use Runtime -> Change runtime type -> GPU.')

from google.colab import drive
drive.mount('/content/drive')


## 2. Clone Or Pull Repository

In [ ]:
REPO_URL = 'https://github.com/neurovlm/neurovlm.git'
REPO_BRANCH = 'neurovlm_gnn'
REPO_DIR = Path('/content/neurovlm')

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=False)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'src'))
print('Repo ready at', REPO_DIR)
print(subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip())


## 3. Install Dependencies

In [ ]:
%pip install -q -e /content/neurovlm
%pip install -q torch pandas numpy nibabel nilearn pyyaml tqdm safetensors transformers adapters pyarrow matplotlib scikit-learn


## 4. Copy Generated Data From Drive Into The Clone

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive')
PREFERRED_DRIVE_PROJECT_DIR = DRIVE_ROOT / 'neurovlm'

required_rel = [
    'atlas_free_multipositive/cache/unified_jsonl/splits/train.jsonl',
    'atlas_free_multipositive/cache/unified_jsonl/splits/val.jsonl',
    'atlas_free_multipositive/cache/unified_jsonl/text_registry.jsonl',
    'atlas_free_multipositive/cache/text_embeddings/specter_text_cache.pt',
    'atlas_free_multipositive/data/ale_caches/atlas_free_4mm_fwhm9_crop_float16.pt',
]

def has_generated_data(root: Path) -> bool:
    return all((root / rel).exists() for rel in required_rel)

candidates = []
if has_generated_data(PREFERRED_DRIVE_PROJECT_DIR):
    candidates.append(PREFERRED_DRIVE_PROJECT_DIR)
for train_json in DRIVE_ROOT.glob('**/atlas_free_multipositive/cache/unified_jsonl/splits/train.jsonl'):
    root = train_json.parents[4]
    if has_generated_data(root) and root not in candidates:
        candidates.append(root)

if not candidates:
    print('Searched for generated data under', DRIVE_ROOT)
    raise FileNotFoundError('Could not find generated atlas_free_multipositive data on Drive. Put it under /content/drive/MyDrive/neurovlm or another folder with the same relative paths.')

DRIVE_PROJECT_DIR = candidates[0]
print('Using generated data from', DRIVE_PROJECT_DIR)

copy_roots = [
    'atlas_free_multipositive/cache',
    'atlas_free_multipositive/data/ale_caches',
]
for rel in copy_roots:
    src = DRIVE_PROJECT_DIR / rel
    dst = REPO_DIR / rel
    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print('copied', src, '->', dst)

DRIVE_RUNS_DIR = DRIVE_PROJECT_DIR / 'atlas_free_multipositive/outputs/runs'
DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)
print('Drive run outputs:', DRIVE_RUNS_DIR)


## 5. Verify Local Training Files

In [ ]:
missing = []
for rel in required_rel:
    p = REPO_DIR / rel
    ok = p.exists()
    print(f'{str(ok):5}  {p}')
    if not ok:
        missing.append(str(p))
if missing:
    raise FileNotFoundError('Missing local files after Drive copy: ' + repr(missing))


## 6. Locate Stage 1/2 Checkpoints

In [ ]:
import yaml
from atlas_free_multipositive.training.train_joint_generation_contrastive import train_from_config as train_joint
manifests = sorted(DRIVE_RUNS_DIR.glob('colab_generation_*/generation_stage_manifest.json'))
if not manifests:
    raise FileNotFoundError('Run 09_10_colab_generation_pipeline.ipynb first so Stage 1/2 checkpoints exist on Drive.')
manifest_path = manifests[-1]
manifest = json.loads(manifest_path.read_text())
AE_CHECKPOINT = manifest['autoencoder_checkpoint']
TEXT_PROJ_CHECKPOINT = manifest['random_text_to_brain_checkpoint']
print('manifest:', manifest_path)
print('AE_CHECKPOINT:', AE_CHECKPOINT)
print('TEXT_PROJ_CHECKPOINT:', TEXT_PROJ_CHECKPOINT)


## 7. Joint Generation + Contrastive Training

In [ ]:
joint_cfg = yaml.safe_load(Path('atlas_free_multipositive/configs/joint_generation_contrastive_config.yaml').read_text())
joint_run_dir = DRIVE_RUNS_DIR / time.strftime('colab_joint_%Y%m%d_%H%M%S')
joint_cfg.update({'autoencoder_checkpoint': AE_CHECKPOINT, 'text_projection_checkpoint': TEXT_PROJ_CHECKPOINT, 'checkpoint_dir': str(joint_run_dir / 'checkpoints'), 'device': 'auto', 'epochs': 3, 'batch_size': 4})
result = train_joint(joint_cfg)
joint_run_dir.mkdir(parents=True, exist_ok=True)
json.dump(result, open(joint_run_dir / 'joint_result.json', 'w'), indent=2)
print(json.dumps(result, indent=2))
